In [1]:
# Simple text wrapping that actually works
import textwrap
import sys
from IPython.core.interactiveshell import InteractiveShell

# Set up text wrapping for output
InteractiveShell.ast_node_interactivity = "all"

# Configure display width 
import os
os.environ['COLUMNS'] = '100'

# Simple string wrapper for long outputs
import builtins

# Store the original repr function
original_repr = builtins.repr

def wrapped_repr(obj):
    """Wrap long text outputs"""
    result = original_repr(obj)
    if len(result) > 100:
        return textwrap.fill(result, width=100, break_long_words=False, break_on_hyphens=False)
    return result

# Override repr globally for text wrapping
builtins.repr = wrapped_repr

print("✅ Simple text wrapping configured!")
print("📝 Long outputs will automatically wrap at 100 characters")


✅ Simple text wrapping configured!
📝 Long outputs will automatically wrap at 100 characters


# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [2]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [3]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Loan Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [4]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/complaints.csv",
    metadata_columns=[
      "Date received", 
      "Product", 
      "Sub-product", 
      "Issue", 
      "Sub-issue", 
      "Consumer complaint narrative", 
      "Company public response", 
      "Company", 
      "State", 
      "ZIP code", 
      "Tags", 
      "Consumer consent provided?", 
      "Submitted via", 
      "Date sent to company", 
      "Company response to consumer", 
      "Timely response?", 
      "Consumer disputed?", 
      "Complaint ID"
    ]
)

loan_complaint_data = loader.load()

for doc in loan_complaint_data:
    doc.page_content = doc.metadata["Consumer complaint narrative"]

Let's look at an example document to see if everything worked as expected!

In [5]:
loan_complaint_data[0]

Document(metadata={'source': './data/complaints.csv', 'row': 0, 'Date received': '03/27/25',
'Product': 'Student loan', 'Sub-product': 'Federal student loan servicing', 'Issue': 'Dealing with
your lender or servicer', 'Sub-issue': 'Trouble with how payments are being handled', 'Consumer
complaint narrative': "The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX.
However, payments were not re-amortized on my federal student loans currently serviced by Nelnet
until very recently. The new payment amount that is effective starting with the XX/XX/XXXX payment
will nearly double my payment from {$180.00} per month to {$360.00} per month. I'm fortunate that my
current financial position allows me to be able to handle the increased payment amount, but I am
sure there are likely many borrowers who are not in the same position. The re-amortization should
have occurred once the forbearance ended to reduce the impact to borrowers.", 'Company public
response': 'None', 'Company'

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "LoanComplaints".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [6]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    loan_complaint_data,
    embeddings,
    location=":memory:",
    collection_name="LoanComplaints"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [7]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [8]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [9]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [10]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [11]:
naive_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issues with loans, based on the complaints provided, tend to involve problems with
how loans are managed or serviced. These issues include errors in loan balances, misapplied
payments, wrongful denials of payment plans, inaccurate or outdated information on credit reports,
difficulties in applying payments correctly, and mishandling of loan transfers or communication
failures. \n\nA recurring theme is borrowers experiencing errors and mismanagement that lead to
incorrect balances, negative credit impacts, or difficulties in repayment, often compounded by lack
of transparency or communication from servicers.\n\nIf you are seeking a specific most common issue
from this dataset, it appears to be related to **"Dealing with your lender or servicer,"**
especially issues like misapplication of payments, errors in balances, or mishandling of account
information. \n\nPlease note that the broader pattern indicates that loan servicing errors and
mismanagement are primary concerns

In [12]:
naive_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

"Based on the provided complaints, yes, some complaints were not handled in a timely manner. For
instance:\n\n- The complaint received on 03/28/25 by MOHELA was marked as 'N/A' for timely response,
indicating it was not addressed promptly.\n- The complaint received on 04/24/25 by Maximus Federal
Services, Inc. was handled 'Yes' for timely response.\n- The complaint received on 04/14/25 by
Nelnet, Inc. was handled 'Yes' for timely response.\n- The complaint received on 04/18/25 by
EdFinancial Services was handled 'Yes' for timely response.\n- The complaint received on 04/05/25 by
Maximus Federal Services, Inc. was handled 'Yes' for timely response.\n- The complaint received on
05/02/25 by EdFinancial Services was handled 'Yes' for timely response.\n\nAdditionally, the
complaint from 03/28/25 regarding a delayed bank account setup by MOHELA specifically mentions an
issue that persisted over 2-3 weeks, indicating delays in handling.\n\nTherefore, at least one
complaint was not handled in 

In [13]:
naive_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for several reasons, including:\n\n1. Lack of clear
communication and timely notification from loan servicers about when repayments would resume or if
there were changes in loan processing, leading to unexpected delinquencies and credit issues.\n2.
Difficulties in managing payment options, such as being limited to forbearance or deferment, which
can cause interest to accumulate and increase the total amount owed over time.\n3. Financial
hardships and economic challenges, including stagnant wages, high living expenses, unemployment, or
working in jobs that do not support loan repayment.\n4. Confusion or misinformation about loan
terms, interest rates, and repayment plans, making it hard for borrowers to navigate their
obligations.\n5. Issues with loan mismanagement, transfer of loans without proper notification, or
problems with the loan servicing process that prevent borrowers from making or adjusting payments
effectively.\n6. Borrowers feeling mi

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [14]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(loan_complaint_data, )

We'll construct the same chain - only changing the retriever.

In [15]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [16]:
bm25_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

"Based on the provided context, the most common issue with loans appears to be problems related to
dealing with lenders or servicers, such as disputes over fees, payment application issues, incorrect
or bad information about loans, and general difficulty in obtaining accurate or satisfactory
responses from the loan servicers. Many complaints highlight issues like charges the consumer does
not agree with, inability to apply payments correctly, receiving incorrect or confusing information
about loan balances, or the servicer's failure to address concerns properly."

In [17]:
bm25_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, all the complaints mentioned in the context received a response
from the companies and were marked as "Closed with explanation," indicating that they were handled
in a timely manner. The responses explicitly state "Timely response?": "Yes" for each complaint.
Therefore, there is no indication that any complaints did not get handled in a timely manner.'

In [18]:
bm25_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans for various reasons, including issues with payment plans,
miscommunication, and handling by loan servicers. Some specific reasons include:\n\n- Being steered
into incorrect types of forbearances or loan deferments, which prevented proper repayment and
increased debt due to capitalized interest.\n- Poor communication from loan servicers, such as not
responding to requests for deferments or forbearances, or failing to notify borrowers about changes
in their loan status.\n- Errors or issues with automatic payments, such as payments being reversed
or canceled without notice, or autopay being discontinued without informing the borrower.\n-
Receiving bad or unclear information about their loans, leading to missed or late payments.\n-
Servicers not responding appropriately to requests for assistance, such as applying for forbearance
or deferment.\n- Unclear or delayed notifications about loan transfers or changes in loan
servicing.\n\nOverall, failures t

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

---

#### ✅ Answer #1:

BM25 would outperform embeddings when specific text matching is involved. Embeddings would potentially struggle because they may not be able to quantify semantic meaning from such a peculiar word like "Mohela".

"Why did the Mohela complaint from late March 2025 fail to get handled in a timely manner?"

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [19]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [20]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [21]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided complaints, the most common issue with loans appears to be problems related
to dealing with lenders or servicers, specifically issues such as receiving bad or incorrect
information about the loan, errors in loan balances, misapplied payments, wrongful denials of
payment plans, and mishandling of loan data. Many complaints also involve lack of communication,
unapproved transfer of loans, and disputes over account discrepancies, indicating that errors and
mismanagement by loan servicers are prevalent issues.'

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Yes, according to the provided complaints, some did not get handled in a timely manner. For
example, one complaint about a student loan issue has been open since an unspecified time (referred
to as "XXXX") and still remains unresolved after nearly 18 months. Similarly, the complaint
regarding unresolved payments and account updates has been ongoing for over 2-3 weeks, and the issue
has not been resolved yet.'

In [23]:
contextual_compression_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans for several reasons, including:\n\n1. Lack of awareness or
understanding: Many borrowers were not informed by their financial aid officers or loan servicers
that they would need to repay their loans, leading to surprise and confusion about repayment
obligations.\n\n2. Problems with communication and notifications: Borrowers reported that they did
not receive proper notifications about when to start paying, changes in loan ownership, or payment
due dates, which contributed to missed payments or late payments.\n\n3. Difficulty managing interest
and repayment options: Borrowers found that options like forbearance or deferment allowed interest
to continue accruing, making it harder to pay off loans later, especially when payments were reduced
or delayed.\n\n4. Financial hardships and stagnant wages: Many borrowers faced financial challenges,
such as not being able to afford increased payments or not qualifying for loan forgiveness programs,
making rep

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [25]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
)

In [26]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [27]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided complaints, appears to be problems related
to handling, servicing, and inaccuracies of student loans. This includes issues such as errors in
loan balances, misapplied payments, wrongful denials of payment plans, lack of transparency, changes
in loan servicing without notice, and challenges in getting accurate information or correcting
account information. Additionally, many complaints highlight difficulties with repayment plans,
interest accumulation during forbearance or pandemic periods, and issues with loan transfer
notifications.\n\nIn summary, the most common issues involve **mismanagement and lack of
transparency by loan servicers, errors in account information, and difficulties in managing
repayment or correcting inaccuracies**.'

In [28]:
multi_query_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, yes, some complaints did not get handled in a timely manner.
Several complaints explicitly state delays or failures to respond within expected timeframes, such
as:\n\n- Complaint #12739706 (MOHELA) was not responded to within the required 30 days, and the
response was delayed by more than 3 hours of wait time.\n- Complaint #12832400 (Maximus) was handled
with a response within the designated timeframe.\n- Complaint #12823876 (EdFinancial) was responded
to promptly as well.\n- Several other complaints mention that the company failed to provide
responses in the promised time or did not respond at all, including complaints that note delays of
over 30 days with no action, or repeated follow-ups with no resolution.\n\nTherefore, the answer is
yes: some complaints did not get handled in a timely manner.'

In [29]:
multi_query_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans for various reasons, including:\n\n- Accumulation of high
interest rates over the years, making the debt grow despite payments.\n- Financial hardship and
income loss, such as homelessness, unemployment, or medical issues.\n- Systemic issues and
mismanagement by servicers, including misapplied payments, errors in loan balances, and failure to
provide proper information about repayment options.\n- Lack of proper communication or notification
from loan servicers about payment due dates, payment resumption, or changes in loan status.\n-
Coercive or deceptive practices, such as being steered into long-term forbearances and
consolidations without being informed of better alternatives like income-driven repayment or
forgiveness programs.\n- Administrative errors, improper reporting to credit bureaus, and systemic
failures that led to delinquency or damage to credit scores.\n- Challenges related to managing
multiple loans or transferring accounts without 

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

---

#### ✅ Answer #2:

Multiple reformulations gives the LLM different ways of saying the same thing or asking the same question using similar words and phrases. It also broadens the scope of semantic coverage, expanding the potential set of content to draw from, potentially improving recall. 

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [30]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = loan_complaint_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [31]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

True

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [32]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [33]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [34]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [35]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the provided complaints, appears to be problems related
to federal student loan servicing. These include errors in loan balances, misapplied payments,
wrongful denials of payment plans, and misconduct by servicers, such as errors in reporting,
unverified debt collection, and issues arising from the transfer or sale of loans. Additionally,
other frequent issues involve incorrect information on credit reports, discrepancies in interest
rates, and challenges in managing or consolidating loans.'

In [36]:
parent_document_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Yes, there are complaints indicating that they were not handled in a timely manner. For example,
complaints with IDs 12709087 and 12935889 explicitly state that responses were "No," meaning they
were not handled promptly. Additionally, the complaint with ID 13205525 was responded to within 30
days, which is generally considered timely. Therefore, at least some complaints did not get handled
in a timely manner.'

In [37]:
parent_document_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans mainly due to a combination of factors such as mismanagement
and lack of clear communication from loan servicing companies, financial hardship, and
misinformation about repayment obligations. For example, some individuals experienced their payments
being resumed or reported as delinquent before completing the grace period, without proper
explanation from the servicers. Others faced severe financial difficulties after attending
underperforming or financially unstable institutions, which made it difficult to secure employment
and manage repayment. Additionally, issues like unverified debts, improper reporting, and failure to
notify borrowers about changes in their loan status also contributed to repayment failures.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [39]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [40]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [41]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issue with loans, based on the context provided, appears to be problems related to
managing and reconciling loan information, such as errors in loan balances, misapplied payments, and
inaccurate reporting. Many complaints highlight difficulties with loan balances not matching
records, incorrect account statuses, and issues with loan transfer or transfer of servicing without
proper notification. Additionally, issues with bad information about loans, the handling or
mishandling of payments, and inadequate communication from loan servicers are prevalent. \n\nIn
summary, the most common issue seems to be the improper handling of loan data leading to
inaccuracies and mismanagement of loan information, which can significantly impact borrowers’ credit
and financial stability.'

In [42]:
ensemble_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints data, several complaints indicate that issues were not handled in
a timely manner. For example:\n\n- One complaint (Row 441, received on 03/28/25) states that the
consumer waited over 15 days for a response after submitting a complaint, and as of the date of the
complaint, no one had reached out. The response was marked as "No" for timely response.\n\n- Another
complaint (Row 503, received on 04/14/25) notes that the company responded "Yes" for timely
response, indicating it was handled within the acceptable timeframe.\n\n- Multiple complaints (e.g.,
Row 468, 418, 95) mention delays of over 10 days or more, despite some being marked as "Timely
response? Yes" or "No". Particularly, Complaint Row 441 explicitly states the complaint was not
handled in a timely manner.\n\nTherefore, the evidence shows that at least some complaints did not
get handled promptly. The complaint from row 441 is a clear example of a complaint that was delayed
beyond the expected

In [43]:
ensemble_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons, including:\n\n1. Lack of adequate
notification and communication from loan servicers about payment due dates, account changes, or
transfer of loans, leading to unawareness of payment obligations.\n2. Difficulty managing payments
due to financial hardship, stagnant wages, or increased interest accumulation, which made repayment
unmanageable.\n3. Mismanagement or misrepresentation by loan servicers, such as steering borrowers
into forbearances with accruing interest or providing bad information about repayment options.\n4.
Unfair practices by servicers, including incorrect reporting of delinquency or default, improper
transfer and handling of loans, and failure to provide proper documentation or support.\n5.
Borrowers experiencing systemic issues like unemployment, health problems, or unexpected expenses
that prevent them from making timely payments.\n6. Difficulty obtaining or navigating repayment
plans, including income-driven

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [44]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [45]:
semantic_documents = semantic_chunker.split_documents(loan_complaint_data[:20])

Let's create a new vector store.

In [46]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Loan_Complaint_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [47]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [48]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [49]:
semantic_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided complaints data, the most common issues with loans, particularly federal
student loans, seem to be related to management and communication problems such as:\n\n- Trouble
with repayment and payment plans (e.g., difficulties with auto-debit, incorrect repayment amounts,
forbearance issues)\n- Problems with loan reporting and credit reporting errors (e.g., accounts in
default when borrower has not been in default, incorrect account status)\n- Lack of transparency and
communication from loan servicers about loan status, issuers, or changes\n- Issues with documenting
and processing loan forgiveness or discharge requests\n- Unauthorized disclosures or breaches of
personal and financial data\n\nOverall, issues involving mismanagement, miscommunication, and errors
in loan reporting are most prevalent. If focusing on the most common specific complaint type, it
appears that borrowers often face difficulties understanding or managing their repayment terms and
account status

In [50]:
semantic_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints, several did not get handled in a timely manner. Specifically, the
complaint regarding Nelnet, Inc. (Row 17) indicates that despite multiple letters and acknowledgment
of receipt, Nelnet never responded to the consumer\'s certified mail and violations of law, which
suggests a lack of timely handling. The complaint was closed with an explanation, but the persistent
lack of response indicates it was not handled promptly.\n\nSimilarly, the complaint about
EdFinancial Services (Row 4) notes that the response time was "Yes" for being timely, but in other
complaints, like the one involving Nelnet (Row 17), there is explicit evidence that some complaints
were not responded to within an adequate or timely manner.\n\nTherefore, yes, some complaints,
particularly the one involving Nelnet\'s failure to respond to multiple certified mail inquiries
despite acknowledgment, did not get handled in a timely manner.'

In [51]:
semantic_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons, including:\n\n- Difficulty dealing with
lenders or servicers, such as receiving bad information or lack of transparency, which causes undue
stress.\n- Problems submitting accurate documentation for loan forgiveness programs, leading to
delays or stalls.\n- Disputes over the legitimacy or accuracy of the loan reports, sometimes due to
improper reporting or unauthorized disclosures.\n- Issues with payment processing, such as payments
not clearing due to bank errors or technical problems.\n- Situations where loans are improperly
reported as in default due to mistakes or breaches in privacy and data handling.\n- Borrowers’
inability to handle increased payments after forbearance ends, especially if re-amortization or
proper notification was not given.\n- Legal disputes over the validity of the loans or claims that
the debts have been illegally reported or are otherwise invalid.\n\nIn summary, failures to pay back
loans can be caus

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

---

#### ✅ Answer #3:

Semantic chunking probably would not behave well with sentences that are short and highly repetitive like FAQs because such semantic similarity would make it difficult to find the places where the chunking needs to occur. We'd likely get a few number of massive chunks or a large number of smaller chunks.

I would adjust the algorithm to use a structure-based chunking for short and highly repetitive sentences like FAQs, because then chunks can be done using sets of Q&A pairs.


# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [ ]:
from ragas.testset import TestsetGenerator

from ragas.metrics import (
    ContextPrecision,
    ContextRecall, 
    ContextRelevance,
    AnswerRelevancy,
    Faithfulness
)
from ragas.llms import LangchainLLMWrapper
from ragas import evaluate
from datasets import Dataset
import pandas as pd
import time
from datetime import datetime

In [60]:
test_documents = loan_complaint_data[:100]
simple = "simple"
reasoning = "reasoning" 
multi_context = "multi_context"

generator = TestsetGenerator.from_langchain(
    chat_model,  # Using the same LLM for generation
    chat_model,  # Using same model for critic as well
    embeddings   # Using our embedding model
)

print("📝 Generating test cases with different evolution types...")

# Generate synthetic test dataset
testset = generator.generate_with_langchain_docs(
    documents=test_documents,
    test_size=20,  # Generate 20 test questions
    distributions={
        simple: 0.5,      # 50% simple questions
        reasoning: 0.25,  # 25% reasoning questions 
        multi_context: 0.25  # 25% multi-context questions
    }
)

print(f"✅ Generated {len(testset)} test cases")
print(f"📊 Test case preview:")
for i, test_case in enumerate(testset[:3]):
    print(f"  Question {i+1}: {test_case.question[:100]}...")
    print(f"  Answer {i+1}: {test_case.ground_truth[:100]}...")
    print("  ---")



📝 Generating test cases with different evolution types...


TypeError: TestsetGenerator.generate_with_langchain_docs() got an unexpected keyword argument 'test_size'. Did you mean 'testset_size'?

In [ ]:
test_documents = loan_complaint_data[:100]

generator = TestsetGenerator.from_langchain(
    chat_model,  # Using the same LLM for generation
    chat_model,  # Using same model for critic as well
    embeddings   # Using our embedding model
)

testset = generator.generate_with_langchain_docs(
    documents=test_documents,
    test_size=20,  # Generate 20 test questions
    distributions={
        simple: 0.5,      # 50% simple questions
        reasoning: 0.25,  # 25% reasoning questions 
        multi_context: 0.25  # 25% multi-context questions
    }
)

print(f"✅ Generated {len(testset)} test cases")
print(f"📊 Test case preview:")
for i, test_case in enumerate(testset[:3]):
    print(f"  Question {i+1}: {test_case.question[:100]}...")
    print(f"  Answer {i+1}: {test_case.ground_truth[:100]}...")
    print("  ---")


In [ ]:
# Step 3: Create Evaluation Framework
print("🔧 Setting up evaluation framework...")

# Define our retriever configurations for testing
retrievers_config = {
    "naive": {
        "retriever": naive_retriever,
        "chain": naive_retrieval_chain,
        "description": "Basic vector similarity search"
    },
    "bm25": {
        "retriever": bm25_retriever, 
        "chain": bm25_retrieval_chain,
        "description": "Keyword-based sparse retrieval"
    },
    "multi_query": {
        "retriever": multi_query_retriever,
        "chain": multi_query_retrieval_chain, 
        "description": "Multiple query reformulations"
    },
    "parent_document": {
        "retriever": parent_document_retriever,
        "chain": parent_document_retrieval_chain,
        "description": "Small-to-big chunk strategy"
    },
    "contextual_compression": {
        "retriever": compression_retriever,
        "chain": contextual_compression_retrieval_chain,
        "description": "Reranking with Cohere"
    },
    "ensemble": {
        "retriever": ensemble_retriever,
        "chain": ensemble_retrieval_chain,
        "description": "Combination of all retrievers"
    }
}

# Set up Ragas evaluator LLM
evaluator_llm = LangchainLLMWrapper(chat_model)

# Initialize Ragas metrics with the evaluator LLM
context_precision = ContextPrecision(llm=evaluator_llm)
context_recall = ContextRecall(llm=evaluator_llm)
context_relevance = ContextRelevance(llm=evaluator_llm)
answer_relevancy = AnswerRelevancy(llm=evaluator_llm)
faithfulness = Faithfulness(llm=evaluator_llm)

# Function to evaluate a single retriever
def evaluate_retriever(name, config, testset, max_samples=5):
    """Evaluate a single retriever with Ragas metrics"""
    print(f"🔍 Evaluating {name} retriever...")
    
    # Prepare data for evaluation
    questions = [t.question for t in testset[:max_samples]]
    ground_truths = [t.ground_truth for t in testset[:max_samples]]
    
    # Get answers and contexts from the retriever chain
    answers = []
    contexts = []
    
    start_time = time.time()
    
    for question in questions:
        try:
            # Get response from chain
            result = config["chain"].invoke({"question": question})
            answers.append(result["response"].content)
            
            # Extract context from retrieved documents
            context_docs = result["context"]
            context_text = [doc.page_content for doc in context_docs]
            contexts.append(context_text)
            
        except Exception as e:
            print(f"❌ Error processing question: {e}")
            answers.append("Error occurred")
            contexts.append(["No context retrieved"])
    
    evaluation_time = time.time() - start_time
    
    # Create dataset for Ragas evaluation using correct field names
    data = {
        "user_input": questions,           # Updated field name
        "response": answers,               # Updated field name  
        "retrieved_contexts": contexts,    # Updated field name
        "reference": ground_truths         # Updated field name
    }
    
    dataset = Dataset.from_dict(data)
    
    # Evaluate with Ragas metrics (focus on retrieval metrics)
    result = evaluate(
        dataset=dataset,
        metrics=[
            context_precision,   # How precise are the retrieved contexts
            context_recall,      # How much of the relevant context was retrieved
            context_relevance,   # How relevant are the contexts to the question
            faithfulness,        # How faithful is the answer to the context
            answer_relevancy     # How relevant is the answer to the question
        ]
    )
    
    return {
        "name": name,
        "description": config["description"],
        "metrics": result,
        "evaluation_time": evaluation_time,
        "samples_evaluated": len(questions)
    }

print("✅ Evaluation framework ready!")


In [ ]:
# Step 4: Run Evaluations for All Retrievers
print("🏃 Running comprehensive evaluation for all retrievers...")
print("⏱️ This may take several minutes due to LLM calls...")

# Store all evaluation results
evaluation_results = []

# Evaluate each retriever
for name, config in retrievers_config.items():
    try:
        result = evaluate_retriever(name, config, testset, max_samples=5)
        evaluation_results.append(result)
        print(f"✅ Completed evaluation for {name}")
        
        # Print quick summary - updated for new Ragas API
        metrics = result["metrics"]
        print(f"   📊 Context Precision: {metrics.get('context_precision', 'N/A')}")
        print(f"   📊 Context Recall: {metrics.get('context_recall', 'N/A')}")
        print(f"   📊 Context Relevance: {metrics.get('context_relevance', 'N/A')}")
        print(f"   ⏱️ Evaluation Time: {result['evaluation_time']:.1f}s")
        print("   ---")
        
    except Exception as e:
        print(f"❌ Failed to evaluate {name}: {e}")
        continue

print(f"🎉 Evaluation complete! Tested {len(evaluation_results)} retrievers.")


In [ ]:
# Step 5: Comprehensive Analysis and Results Compilation
print("📈 Compiling comprehensive analysis...")

# Create detailed comparison table - updated for new Ragas API
results_df = pd.DataFrame([
    {
        "Retriever": result["name"],
        "Description": result["description"],
        "Context Precision": result["metrics"].get("context_precision", 0.0),
        "Context Recall": result["metrics"].get("context_recall", 0.0), 
        "Context Relevance": result["metrics"].get("context_relevance", 0.0),
        "Faithfulness": result["metrics"].get("faithfulness", 0.0),
        "Answer Relevancy": result["metrics"].get("answer_relevancy", 0.0),
        "Avg Latency (s)": result["evaluation_time"] / result["samples_evaluated"],
        "Total Time (s)": result["evaluation_time"]
    }
    for result in evaluation_results
])

# Sort by overall performance (weighted average of key metrics)
results_df["Overall Score"] = (
    results_df["Context Precision"] * 0.25 +
    results_df["Context Recall"] * 0.25 + 
    results_df["Context Relevance"] * 0.25 +
    results_df["Answer Relevancy"] * 0.25
)

results_df = results_df.sort_values("Overall Score", ascending=False)

print("🏆 RETRIEVER EVALUATION RESULTS")
print("=" * 80)
print(results_df.round(3).to_string(index=False))
print("\n")

# Cost Analysis (estimated based on typical pricing)
cost_estimates = {
    "naive": {"api_calls": 1, "embedding_calls": 1, "complexity": "Low"},
    "bm25": {"api_calls": 1, "embedding_calls": 0, "complexity": "Low"}, 
    "multi_query": {"api_calls": 4, "embedding_calls": 4, "complexity": "Medium"},
    "parent_document": {"api_calls": 1, "embedding_calls": 2, "complexity": "Medium"},
    "contextual_compression": {"api_calls": 2, "embedding_calls": 1, "complexity": "High"},
    "ensemble": {"api_calls": 6, "embedding_calls": 6, "complexity": "Very High"}
}

print("💰 COST & COMPLEXITY ANALYSIS")
print("=" * 50)
for name, costs in cost_estimates.items():
    print(f"{name:20}: API calls/query: {costs['api_calls']}, "
          f"Embedding calls: {costs['embedding_calls']}, "
          f"Complexity: {costs['complexity']}")

print("\n🏅 FINAL RECOMMENDATION")
print("=" * 50)

# Get top performer
best_retriever = results_df.iloc[0]
print(f"🥇 Best Overall Performance: {best_retriever['Retriever']}")
print(f"   Overall Score: {best_retriever['Overall Score']:.3f}")
print(f"   Context Precision: {best_retriever['Context Precision']:.3f}")
print(f"   Context Recall: {best_retriever['Context Recall']:.3f}")
print(f"   Average Latency: {best_retriever['Avg Latency (s)']:.2f}s")

# Best value (good performance vs cost)
results_df["Value Score"] = results_df["Overall Score"] / results_df["Avg Latency (s)"]
best_value = results_df.loc[results_df["Value Score"].idxmax()]
print(f"\n🎯 Best Value (Performance/Latency): {best_value['Retriever']}")
print(f"   Value Score: {best_value['Value Score']:.3f}")

print("\n📝 DETAILED ANALYSIS:")
print("=" * 50)

analysis_text = f"""
Based on the comprehensive evaluation of {len(evaluation_results)} retrieval methods on the loan complaints dataset, 
the **{best_retriever['Retriever']} retriever** emerges as the top performer with an overall score of {best_retriever['Overall Score']:.3f}.

**Performance Insights:**
- **Context Precision**: {best_retriever['Retriever']} achieved {best_retriever['Context Precision']:.3f}, indicating high accuracy in retrieving relevant documents
- **Context Recall**: Score of {best_retriever['Context Recall']:.3f} shows good coverage of relevant information  
- **Latency**: Average query time of {best_retriever['Avg Latency (s)']:.2f}s provides acceptable response times

**Cost Considerations:**
- BM25 offers the lowest cost (no embedding calls) but potentially lower semantic understanding
- Multi-query and ensemble methods provide robust performance but at 4-6x the API cost
- Contextual compression balances performance with moderate cost overhead

**For this loan complaints dataset specifically:**
The structured nature of complaint data with clear topics (payment issues, servicer problems, etc.) 
benefits from the {best_retriever['Retriever']} approach because it {
    'captures semantic relationships well' if 'naive' in best_retriever['Retriever'] 
    else 'combines multiple retrieval strategies effectively' if 'ensemble' in best_retriever['Retriever']
    else 'provides sophisticated query understanding' if 'multi' in best_retriever['Retriever']
    else 'efficiently reranks results for relevance' if 'compression' in best_retriever['Retriever']
    else 'leverages keyword matching for specific complaint types' if 'bm25' in best_retriever['Retriever']
    else 'maintains context while improving precision'
}. 

**Recommendation**: For production deployment, consider the {best_value['Retriever']} retriever as it provides 
the best performance-to-cost ratio, making it optimal for high-volume complaint processing systems.
"""

print(analysis_text)

# Save results to update TODO
print("\n✅ Evaluation complete and analysis generated!")
